In [ ]:
from collections import defaultdict
from copy import deepcopy
import glob
import json
import os
import re

import numpy as np
import pandas as pd

os.chdir('C:\\Users\\hlloyd\\projects\\union-lists')
!uv run python -c "from union_lists.transform import reformat_union_lists as ref"
os.chdir('notebooks')

### Import csv and metadata files

In [ ]:
csv_files = glob.glob("../data/interim/Half Inch/*.csv")
metadata_files = glob.glob("../data/interim/Half Inch/*.json")
# docx_files = [x for x in docx_files if "~" not in x and "(2)" not in x]
# docx_files = [x for x in docx_files if "_mod" not in x]

In [ ]:
csv_files, metadata_files

In [ ]:
def pre_process_df(df: pd.DataFrame) -> pd.DataFrame:
    # Apply any preprocessing that can be column vectorised before the rows are processed
    for col in df.dropna(axis=1, how="all"):
        if df[col].dtype in (pd.StringDtype(na_value=np.nan), str):
            df[col] = df[col].str.replace("  ", " ")
    return df

In [ ]:
dfs, metadatas = {}, {}
for f in csv_files[:-2]:
    file_id = os.path.basename(f).split(".")[0] + ".doc"
    df = pd.read_csv(f, encoding="utf8")
    with open(f[:-4] + ".json") as g:
        metadata = json.load(g)
    dfs[file_id] = pre_process_df(df)
    metadatas[file_id] = metadata

In [ ]:
entry_dfs = {}
for file_id, df in dfs.items():
    print(file_id)
    entries = []
    [entries.extend(process_6col_row(row[1], source=file_id, scale="Half Inch", metadata=metadatas[file_id])) for row in df.iterrows()];
    entry_df = pd.concat([pd.DataFrame(x, index=[0]) for x in entries]).reset_index(drop=True)
    entry_dfs[file_id] = entry_df

In [ ]:
sample_df = pd.concat([entry_dfs["34B.doc"], entry_dfs["35B.doc"]])
# sample_df.to_csv("../data/processed/two_doc_sample.csv", encoding="utf-8-sig", index=False)
# sample_df.to_excel("../data/processed/two_doc_sample.xlsx", index=False)

In [ ]:
# This sort_values arranges Half Inch quadrants in the current BL cataloguing order of NW/NE/SW/SE compared to the previous standard of NW/SW/NE/SE
# I haven't implemented it as it makes it harder to map results to the raw data by row

# .sort_values(by=["Post-1905 Block Number", "Post-1905 Block Letter", "Post-1905 Sheet ID"], key=lambda x: x.apply(lambda y:{"NW":1, "SW":3, "NE":2, "SE":4}.get(y, y)))

In [ ]:
sample_df

In [ ]:
sources_by_scale = {
    "63,360": None,
    "1:126,720": set([os.path.basename(x) for x in glob.glob("C:\\Users\\hlloyd\\projects\\union-lists\\data\\raw\\Half Inch\\*.doc")]),
    "1:253,440": None
}

In [ ]:
assert len(sample_df["Scale"].unique()) == 1  # Won't work on a concatenated df of all the individual doc dfs
assert set(sample_df["Source File"].unique()) <= sources_by_scale[sample_df["Scale"].unique()[0]]
assert np.array_equal(sample_df["Location Room"].unique(), ["UGF"])
assert set(sample_df["Time Period"].unique()) <= {"<1886", "1886-1905", "1905>", None}

full_table_text = entry_dfs["38B"].dropna(axis=1, how="all").sum().sum()
ref_re = re.compile(r"X/\d{1,5}/\w{1,4}/\w{1,4} {1}[\d/]{4,9}")
assert full_table_text.count("X/") == len(ref_re.findall(full_table_text))
full_refs_with_date = (sample_df["Full Reference"].str.lstrip("IOR/") + " " + (sample_df["Edition Date"] + "/" + sample_df["Print Date"]).str.lstrip("/")).dropna()
assert set(full_refs_with_date) == set(ref_re.findall(full_table_text))
assert np.array_equal(sample_df["Parent Reference"].dropna().str.count("/").unique(), [2.0])

all_xnums = dfs["38B"].astype(str).sum(axis=1).str.replace("  ", " ").apply(lambda x: set(ref_re.findall(x)))
related_lookup = defaultdict(set)
for xnums in all_xnums:
    for xnum in xnums:
        related_lookup[xnum] |= xnums
for k,v in related_lookup.items():
    v -= {k}

for k, ref in full_refs_with_date.items():
    output_related_refs = set(("\n".join(sample_df.loc[k, ["Post-1905 Related References", "1886-1905 Related References", "Pre-1886 Related References"]])).split("\n")) - {""}
    assert output_related_refs <= related_lookup[ref]

assert sample_df[
    [
        'Related Sheet', 'Sheet Title', 'Edition Number', 
        'Designation_1', 'Designation_2', 'Publication Date', 'Print Reference', 'Copies Printed',
        'Coloured', 'Gridded', 'Number of Copies', 'Repmat', 'Latitude', 'Longitude', 'Available'
    ]
].dropna(axis=1, how="all").empty

In [ ]:
# sample_df.to_csv("../data/processed/v0.4_sample.csv", encoding="utf-8-sig", index=False)

In [ ]:
{
    "Series Title": f"Survey of India India and Adjacent Countries scale Series",
    "Scale": {"Quarter Inch": "1:253,440", "Half Inch": "1:126,720", "One Inch": "63,360"},
    "Published": None,  # How to tell if published and we don't have a copy? See Huw's answer in OneNote - by reference to e.g. X9052 docs which have more info
    "Location Room": "UGF",
    "Location Section": None,  # dict lookup with external resource, out of scope currently, see Issue #2
    "Location Detail": None,  # dict lookup with external resource, out of scope currently, see Issue #2
    "Full Reference": None,
    "Print Date": None,
    "Time Period": None,
    "Parent Reference": None,
    "Post-1905 Related References": "", # Introduced with Issue #3
    "1886-1905 Related References": "", # Introduced with Issue #3
    "Pre-1886 Related References": "", # Introduced with Issue #3
    "Post-1905 Block Number": None,
    "Post-1905 Block Letter": None,
    "Post-1905 Sheet ID": None,
    "1886-1905 New Sheet ID": None,
    "1886-1905 Old Sheet ID": None,
    "Pre-1886 New Sheet ID": None,
    "Pre-1886 Old Sheet ID": None,
    "Related Sheet": None,
    "Sheet Title": None,
    "Edition Number": None,
    "Edition Date": None,
    "Designation_1": None,
    "Designation_2": None,
    "Publication Date": None,
    "Print Reference": None,
    "Copies Printed": None,
    "Coloured": None,
    "Gridded": None,
    "Number of Copies": None,
    "Repmat": None,
    "Latitude": None,
    "Longitude": None,
    "Available": None,
    "Notes": ""
};

Design decisions
 - All locations are UGF, the Section/Detail to be added cross-referencing other sources at a later date
 - All references contain "X/"
 - All references produce a separate row in the data format
 - Time period is set by the period a reference comes from
 - If the final part of a reference contains multiple dates separated by a "/" then the second date is taken as the print year and this text is added to Notes: "Print Date: {year}. Reference in source {source} indicates earlier Edition Date: {reference}"
     - The first date will be kept in Edition Year for now so that I can use it to reconstruct full references easily
 - Any line after a reference that does not contain "X/" is a note about the preceding reference and the following text is added to Notes: f"Reference coverage note: {reference}\n"
 - Allow repeat X nums + print dates - this is ok if include post-1905 block number/letter/sheet ID as part of the unique key. this is a bit of a fudge for now as really x num + print date should be the only primary key and should find a way to allow one 1886-1905 x num to relate to multiple post1905 block number/letter/sheet id
 - Check related references as a group, rather than checking that the related references are also in the right time period column. Don't think there's a way for me to work out just by looking at a reference what time period it should be in from the print date
 - Replace all double spaces between refs and dates with single spaces
 - 35B.docx has some double references e.g. X/9935/2/378+379  1894/1909, treat these as one.
 - References with multiple '+' are also treated as a single reference rather than split into multiple e.g. X/9884/2/3+4+10+11

In [ ]:
dfs["38B"].head(20)

### Examine data model

Used to create the fields in process_6col_row

In [ ]:
data_model_df = pd.read_excel("../data/external/Data Model - Draft - for SI Union List - Populated.xlsx", sheet_name="Half Inch")
notes = data_model_df.iloc[0, :].to_dict()
data_model_df = data_model_df.drop(index=0).reset_index(drop=True)

In [ ]:
data_model_df.head(12)

In [ ]:
data_model_df.info()

### Work using old Maps IS file

In [ ]:
is_dtype = {"Block Number": "Int64", "Sheet Number": "Int64", "Date.1": "str", "Date.2": "str", "Date.3": "str", "Drawer": "Int64"}
is_df = pd.read_excel("../data/raw/Maps IS.xlsx", sheet_name="54 RAW", skiprows=5, header=None, names=["Block Number", "Block Letter", "Sheet Number", "Date.1", "Date.2", "Date.3", "Drawer", "Shelfmark", "Extra Date.1", "Extra Date.2"], dtype=is_dtype)
is_df.drop(index=0, inplace=True)
is_gt_df = pd.read_excel("../data/raw/Maps IS.xlsx", sheet_name="54 NEW")

x9_df = pd.read_excel("../data/raw/X_9053.xlsx", sheet_name="54 Raw", skiprows=6, skipfooter=4, header=None, usecols=[0,1], names=["Block String", "Full Reference"])
x9_gt_df = pd.read_excel("../data/raw/X_9053.xlsx", sheet_name="54 New")

In [ ]:
data_model_df.columns

In [ ]:
is_df.columns

In [ ]:
include_if_present = ["Date.2", "Date.3", "Extra Date.1", "Extra Date.2"]
by_date = [is_df.drop(columns=include_if_present).rename(columns={"Date.1": "Date"})]
for col in include_if_present:
    by_date.append(is_df[["Block Number", "Block Letter", "Sheet Number", "Drawer", "Shelfmark"] + [col]].dropna(subset=col).rename(columns={col: "Date"}))

In [ ]:
clean_is_df = pd.concat(by_date).sort_values(["Block Number", "Block Letter", "Sheet Number"]).reset_index(drop=True)
clean_is_df["Village Boundaries"] = clean_is_df["Date"].str.contains("vb")
clean_is_df["Date"] = clean_is_df["Date"].str.strip(" vb").str.strip("&")

In [ ]:
clean_is_df

In [ ]:
expected_output_length = is_df.shape[0] + is_df.dropna(subset="Date.2").shape[0] + is_df.dropna(subset="Date.3").shape[0] + is_df.dropna(subset="Extra Date.1").shape[0] + is_df.dropna(subset="Extra Date.2").shape[0]

In [ ]:
assert expected_output_length == clean_is_df.shape[0]
assert expected_output_length == is_gt_df.shape[0]

In [ ]:
is_gt_df.head()

In [ ]:
x9_df.head()

In [ ]:
x9_df.shape

In [ ]:
pd.isna(x9_df.head().loc[2, "Full Reference"])

In [ ]:
def split_block(s: str|pd.NA) -> (str, str, str)|(pd.NA, pd.NA, pd.NA):
    if pd.isna(s):
        return (pd.NA, pd.NA, pd.NA)
        
    n = s.split("/")[0][:-1]
    l = s.split("/")[0][-1]
    sn = s.split("/")[1]
    return (n, l, sn)

In [ ]:
def split_ref(s: str|pd.NA) -> str|pd.NA:
    if pd.isna(s):
        return pd.NA

    date = s.split()[-1]
    return date

In [ ]:
clean_x9_df = pd.DataFrame(columns=data_model_df.columns)
clean_x9_df["Full Reference"] = x9_df["Full Reference"]
x9_df["Block Split"] = x9_df["Block String"].apply(lambda x: split_block(x))
clean_x9_df["Block Number"] = x9_df["Block Split"].apply(lambda x: x[0])
clean_x9_df["Block Letter"] = x9_df["Block Split"].apply(lambda x: x[1])
clean_x9_df["Sheet Number"] = x9_df["Block Split"].apply(lambda x: x[2])
clean_x9_df["Publication Date"] = clean_x9_df["Full Reference"].apply(lambda x: split_ref(x))

In [ ]:
clean_x9_df.head()

In [ ]:
x9_gt_df.head()